# Evaluation of Gemini 2.5 Pro on Stable Matching Generation

This notebook evaluates the performance of an advanced reasoning model (Gemini 2.5 Pro) on the stable matching generation task.

Experiments are conducted on Impartial Culture (IC) datasets across three difficulty levels:
- Easy (n = 5)
- Medium (n = 20)
- Hard (n = 50)

Each generated matching is evaluated for:
- Parsing correctness
- Validity (one-to-one matching)
- Stability (absence of blocking pairs)
- Exact match with the ground-truth solution

In [1]:
# =========================================
# phase 1 imports setup and data loading
# =========================================

import os
import re
import json
import pandas as pd


In [2]:
# ==================================================
# phase 1b gemini api setup
# ==================================================

!pip install -q -U google-generativeai

import google.generativeai as genai
from getpass import getpass

# enter api key securely
GEMINI_API_KEY = getpass("enter gemini api key: ")
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

# configure gemini
genai.configure(api_key=os.environ["GEMINI_API_KEY"])

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


enter gemini api key: ··········


In [3]:
# =========================================
# phase 2 clone repo and set data path
# =========================================

REPO_URL = "https://github.com/SamarthKhanna/LLM_Matching_Markets.git"
REPO_DIR = "LLM_Matching_Markets"
DATA_DIR = os.path.join(REPO_DIR, "instances_matchings")

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}

print("repo dir:", REPO_DIR)
print("data dir:", DATA_DIR)
print("csv files:", os.listdir(DATA_DIR))


# =========================================
# phase 3 load csv file
# =========================================

def load_matching_csv(csv_file, base_path=DATA_DIR):
    full_path = os.path.join(base_path, csv_file)
    df = pd.read_csv(full_path)
    return df


# =========================================
# phase 4 infer expected size
# =========================================

def infer_expected_size(csv_file=None, row=None):
    if csv_file is not None:
        match = re.match(r"(\d+)_", csv_file)
        if match:
            return int(match.group(1))

    if row is not None:
        return len(row["man_pref_string"].split("\n"))

    raise ValueError("could not infer expected size")


# =========================================
# phase 5 prepare one instance
# =========================================

def prepare_instance(row):
    men_prefs_raw = row["man_pref_string"].split("\n")
    women_prefs_raw = row["woman_pref_string"].split("\n")

    men_prefs = [list(map(int, line.split(","))) for line in men_prefs_raw]
    women_prefs = [list(map(int, line.split(","))) for line in women_prefs_raw]

    men_lines = []
    for i, prefs in enumerate(men_prefs):
        men_lines.append(f"M{i+1}: [" + ",".join([f"W{w}" for w in prefs]) + "]")

    women_lines = []
    for i, prefs in enumerate(women_prefs):
        women_lines.append(f"W{i+1}: [" + ",".join([f"M{m}" for m in prefs]) + "]")

    men_text = ",\n".join(men_lines)
    women_text = ",\n".join(women_lines)

    ground_truth = row["men_opt"]

    return men_prefs, women_prefs, men_text, women_text, ground_truth


# =========================================
# phase 6 build answer template
# =========================================

def build_answer_template(expected_size):
    answer_lines = []

    for i in range(1, expected_size + 1):
        comma = "," if i < expected_size else ""
        answer_lines.append(f'  "M{i}": "<woman matched with M{i}>"{comma}')

    return "\n".join(answer_lines)


repo dir: LLM_Matching_Markets
data dir: LLM_Matching_Markets/instances_matchings
csv files: ['5_womanmaster_processed.csv', '20_ic_processed.csv', '50_ic_processed.csv', '10_ic_processed.csv', '5_ic_processed.csv', '50_womanmaster_processed.csv', '10_womanmaster_processed.csv', '20_womanmaster_processed.csv']



# 1. Stable Matching Evaluation Pipeline


In [4]:
# =========================================
# 1.1 build prompt
# =========================================

def build_prompt(men_text, women_text, expected_size):
    answer_template = build_answer_template(expected_size)

    prompt = f"""
You are an intelligent assistant who is an expert in algorithms. Consider the following instance of the two-sided matching problem, where {expected_size} men are to be matched with {expected_size} women.

Here are the preference lists for all individuals:

<preferences>
{{
M: {{
{men_text}
}},
W: {{
{women_text}
}}
}}
</preferences>

Your task is to find the proposer-optimal stable matching.

Return ONLY the final matching.
Do NOT provide Python code.
Do NOT explain your reasoning.
Do NOT describe the algorithm.
Your final response must contain only one JSON object enclosed in <answer></answer> tags.

<answer>
{{
{answer_template}
}}
</answer>

Make sure that each man is matched with exactly ONE woman and each woman is matched with exactly ONE man.
""".strip()

    return prompt

# =========================================
# 1.2 parse model response
# =========================================

def normalize_agent_label(label, prefix):
    label = str(label).strip().upper()
    match = re.match(rf"{prefix}\s*(\d+)$", label)
    if match:
        return f"{prefix}{match.group(1)}"
    return label

def normalize_matching_dict(matching):
    normalized = {}

    for k, v in matching.items():
        man = normalize_agent_label(k, "M")
        woman = normalize_agent_label(v, "W")
        normalized[man] = woman

    return normalized


def extract_matching_from_response(raw_output):
    answer_match = re.search(
        r"<answer>\s*(\{[\s\S]*?\})\s*</answer>",
        raw_output,
        re.IGNORECASE
    )

    if answer_match:
        candidate_json = answer_match.group(1)
        try:
            parsed = json.loads(candidate_json)
            return normalize_matching_dict(parsed), True, "parsed from answer tags"
        except Exception as e:
            return None, False, f"json parse error in answer tags: {str(e)}"

    matches = re.findall(r"\{[\s\S]*?\}", raw_output)

    for block in reversed(matches):
        cleaned = block.replace("'", '"')
        cleaned = re.sub(r",\s*}", "}", cleaned)

        try:
            parsed = json.loads(cleaned)
            return normalize_matching_dict(parsed), True, "parsed from fallback json"
        except:
            continue

    return None, False, "no valid json object found"




# =========================================
# 1.3 evaluate matching
# =========================================

def convert_prefs_to_dict(prefs, side_prefix, other_prefix):
    converted = {}

    for i, pref_list in enumerate(prefs, start=1):
        converted[f"{side_prefix}{i}"] = [f"{other_prefix}{x}" for x in pref_list]

    return converted

def prefers(preference_list, a, b):
    return preference_list.index(a) < preference_list.index(b)

def check_validity(llm_matching, expected_size):
    if llm_matching is None:
        return False, "no matching parsed"

    expected_men = {f"M{i}" for i in range(1, expected_size + 1)}
    expected_women = {f"W{i}" for i in range(1, expected_size + 1)}

    if set(llm_matching.keys()) != expected_men:
        return False, "wrong or incomplete men labels"

    assigned_women = list(llm_matching.values())

    if len(assigned_women) != expected_size:
        return False, "wrong number of assignments"

    if not set(assigned_women).issubset(expected_women):
        return False, "invalid women labels"

    if len(set(assigned_women)) != expected_size:
        return False, "duplicate women assigned"

    return True, "valid one-to-one matching"

def check_stability(matching, men_prefs, women_prefs):
    men_prefs_dict = convert_prefs_to_dict(men_prefs, "M", "W")
    women_prefs_dict = convert_prefs_to_dict(women_prefs, "W", "M")

    reverse_matching = {w: m for m, w in matching.items()}
    blocking_pairs = []

    for m in men_prefs_dict:
        current_w = matching[m]

        for w in men_prefs_dict[m]:
            if w == current_w:
                break

            current_m_for_w = reverse_matching[w]

            if prefers(women_prefs_dict[w], m, current_m_for_w):
                blocking_pairs.append((m, w))

    is_stable = len(blocking_pairs) == 0
    return is_stable, blocking_pairs

def parse_ground_truth_pairs(ground_truth_string):
    pairs = re.findall(r"\[M(\d+),\s*W(\d+)\]", ground_truth_string)
    return {f"M{m}": f"W{w}" for m, w in pairs}

def exact_match_with_ground_truth(llm_matching, ground_truth_string):
    ground_truth_dict = parse_ground_truth_pairs(ground_truth_string)
    return llm_matching == ground_truth_dict, ground_truth_dict

def evaluate_matching(llm_matching, men_prefs, women_prefs, ground_truth, expected_size):
    result = {
        "parsed_successfully": llm_matching is not None,
        "is_valid": False,
        "is_stable": False,
        "exact_match": False,
        "validity_message": "",
        "blocking_pairs": [],
        "ground_truth_dict": None
    }

    if llm_matching is None:
        result["validity_message"] = "no matching parsed"
        return result

    is_valid, validity_message = check_validity(llm_matching, expected_size)
    result["is_valid"] = is_valid
    result["validity_message"] = validity_message

    if not is_valid:
        return result

    is_stable, blocking_pairs = check_stability(llm_matching, men_prefs, women_prefs)
    result["is_stable"] = is_stable
    result["blocking_pairs"] = blocking_pairs

    exact_match, ground_truth_dict = exact_match_with_ground_truth(llm_matching, ground_truth)
    result["exact_match"] = exact_match
    result["ground_truth_dict"] = ground_truth_dict

    return result


# =========================================
# 1.4 process one raw response
# =========================================

def process_model_response(
    raw_output,
    men_prefs,
    women_prefs,
    ground_truth_string,
    expected_size,
    show_output=True
):
    parsed_matching, parsed_ok, parse_msg = extract_matching_from_response(raw_output)

    if not parsed_ok:
        result = {
            "parsed_ok": False,
            "parse_msg": parse_msg,
            "parsed_matching": None,
            "evaluation": {
                "parsed_successfully": False,
                "is_valid": False,
                "is_stable": False,
                "exact_match": False,
                "validity_message": parse_msg,
                "blocking_pairs": [],
                "ground_truth_dict": None
            }
        }
    else:
        evaluation = evaluate_matching(
            parsed_matching,
            men_prefs,
            women_prefs,
            ground_truth_string,
            expected_size
        )

        result = {
            "parsed_ok": True,
            "parse_msg": parse_msg,
            "parsed_matching": parsed_matching,
            "evaluation": evaluation
        }

    if show_output:
        eval_result = result["evaluation"]

        print("parsed:", result["parsed_ok"])
        print("valid:", eval_result["is_valid"])
        print("stable:", eval_result["is_stable"])
        print("exact match:", eval_result["exact_match"])
        print("validity message:", eval_result["validity_message"])
        print("blocking pairs count:", len(eval_result["blocking_pairs"]))

        if len(eval_result["blocking_pairs"]) > 0:
            print("blocking pairs:", eval_result["blocking_pairs"])

        print()

    return result



In [11]:
# =========================================
# 1.5 summarize results
# =========================================

def summarize_results(compact_results):
    total_instances = len(compact_results)

    if total_instances == 0:
        return {
            "total_instances": 0,
            "parsed_count": 0,
            "valid_count": 0,
            "stable_count": 0,
            "exact_match_count": 0,
            "avg_blocking_pairs": 0.0
        }

    parsed_count = sum(x["parsed_ok"] for x in compact_results)
    valid_count = sum(x["is_valid"] for x in compact_results)
    stable_count = sum(x["is_stable"] for x in compact_results)
    exact_match_count = sum(x["exact_match"] for x in compact_results)
    avg_blocking_pairs = sum(x["blocking_pairs_count"] for x in compact_results) / total_instances

    return {
        "total_instances": total_instances,
        "parsed_count": parsed_count,
        "valid_count": valid_count,
        "stable_count": stable_count,
        "exact_match_count": exact_match_count,
        "avg_blocking_pairs": round(avg_blocking_pairs, 2)
    }


# =========================================
# 1.6 shared batch runner
# =========================================

def run_model_on_instances(model, csv_file, num_instances=5, start_index=0, num_examples_to_show=2):
    df = load_matching_csv(csv_file)
    expected_size = infer_expected_size(csv_file=csv_file)

    end_index = min(start_index + num_instances, len(df))

    detailed_results = []
    compact_results = []

    for idx in range(start_index, end_index):
        row = df.iloc[idx]

        men_prefs, women_prefs, men_text, women_text, ground_truth = prepare_instance(row)
        prompt = build_prompt(men_text, women_text, expected_size)

        response = model.invoke(prompt)

        # only print first num_examples_to_show instances
        should_print = (idx - start_index) < num_examples_to_show

        result = process_model_response(
            raw_output=response.content,
            men_prefs=men_prefs,
            women_prefs=women_prefs,
            ground_truth_string=ground_truth,
            expected_size=expected_size,
            show_output=should_print
        )

        evaluation = result["evaluation"]

        detailed_results.append({
            "instance_idx": idx,
            "raw_response": response.content,
            "parsed_matching": result["parsed_matching"],
            "evaluation": evaluation
        })

        compact_results.append({
            "instance_idx": idx,
            "parsed_ok": result["parsed_ok"],
            "is_valid": evaluation["is_valid"],
            "is_stable": evaluation["is_stable"],
            "exact_match": evaluation["exact_match"],
            "blocking_pairs_count": len(evaluation["blocking_pairs"])
        })

    summary = summarize_results(compact_results)
    print(summary)

    return detailed_results, compact_results, summary


# ==================================================
# 1.7 advanced model gemini wrapper
# ==================================================

import time

def call_gemini(prompt, model_name="gemini-2.5-pro", sleep_seconds=2):
    try:
        model = genai.GenerativeModel(model_name)
        response = model.generate_content(prompt)

        text = ""

        if hasattr(response, "text") and response.text:
            text = response.text
        elif hasattr(response, "candidates") and response.candidates:
            parts = []
            for candidate in response.candidates:
                content = getattr(candidate, "content", None)
                if content and hasattr(content, "parts"):
                    for part in content.parts:
                        if hasattr(part, "text") and part.text:
                            parts.append(part.text)
            text = "\n".join(parts)

        time.sleep(sleep_seconds)
        return text if text else None

    except Exception as e:
        print("gemini api error:", e)
        return None


def advanced_model_gemini(csv_file, num_instances=5, start_index=0, num_examples_to_show=2):

    df = load_matching_csv(csv_file)
    end_index = min(start_index + num_instances, len(df))

    detailed_results = []

    summary_stats = {
        "total_instances": 0,
        "parsed_count": 0,
        "valid_count": 0,
        "stable_count": 0,
        "exact_match_count": 0,
        "total_blocking_pairs": 0
    }

    for idx in range(start_index, end_index):

        row = df.iloc[idx]

        men_prefs, women_prefs, men_text, women_text, ground_truth = prepare_instance(row)
        expected_size = len(men_prefs)

        prompt = build_prompt(men_text, women_text, expected_size)

        raw_output = call_gemini(prompt, model_name="gemini-2.5-pro")

        if raw_output is None:
         raw_output = ""

        result = process_model_response(
            raw_output,
            men_prefs,
            women_prefs,
            ground_truth,
            expected_size=expected_size,
            show_output=((idx - start_index) < num_examples_to_show)
        )

        eval_result = result["evaluation"]

        summary_stats["total_instances"] += 1

        if result["parsed_ok"]:
            summary_stats["parsed_count"] += 1

        if eval_result["is_valid"]:
            summary_stats["valid_count"] += 1

        if eval_result["is_stable"]:
            summary_stats["stable_count"] += 1

        if eval_result["exact_match"]:
            summary_stats["exact_match_count"] += 1

        summary_stats["total_blocking_pairs"] += len(eval_result["blocking_pairs"])

        detailed_results.append({
            "instance_idx": idx,
            "raw_response": raw_output,
            "parsed_matching": result["parsed_matching"],
            "evaluation": eval_result
        })

    avg_blocking_pairs = (
        summary_stats["total_blocking_pairs"] / summary_stats["total_instances"]
        if summary_stats["total_instances"] > 0 else 0.0
    )

    summary = {
        "total_instances": summary_stats["total_instances"],
        "parsed_count": summary_stats["parsed_count"],
        "valid_count": summary_stats["valid_count"],
        "stable_count": summary_stats["stable_count"],
        "exact_match_count": summary_stats["exact_match_count"],
        "avg_blocking_pairs": round(avg_blocking_pairs, 2)
    }

    return detailed_results, summary_stats, summary

In [6]:
gemini_detailed_5_ic, gemini_results_5_ic, gemini_summary_5_ic = advanced_model_gemini(
    csv_file="5_ic_processed.csv",
    num_instances=5,
    num_examples_to_show=1
)

print(gemini_summary_5_ic)

parsed: True
valid: True
stable: True
exact match: True
validity message: valid one-to-one matching
blocking pairs count: 0

{'total_instances': 5, 'parsed_count': 5, 'valid_count': 5, 'stable_count': 5, 'exact_match_count': 5, 'avg_blocking_pairs': 0.0}


In [7]:
gemini_detailed_20_ic, gemini_results_20_ic, gemini_summary_20_ic = advanced_model_gemini(
    csv_file="20_ic_processed.csv",
    num_instances=5,
    num_examples_to_show=1
)

print(gemini_summary_20_ic)

parsed: True
valid: True
stable: True
exact match: True
validity message: valid one-to-one matching
blocking pairs count: 0

{'total_instances': 5, 'parsed_count': 5, 'valid_count': 5, 'stable_count': 5, 'exact_match_count': 5, 'avg_blocking_pairs': 0.0}


In [12]:
gemini_detailed_50_ic, gemini_results_50_ic, gemini_summary_50_ic = advanced_model_gemini(
    csv_file="50_ic_processed.csv",
    num_instances=1,
    num_examples_to_show=1
)

print(gemini_summary_50_ic)

parsed: True
valid: True
stable: False
exact match: False
validity message: valid one-to-one matching
blocking pairs count: 95
blocking pairs: [('M6', 'W7'), ('M9', 'W38'), ('M11', 'W17'), ('M12', 'W26'), ('M12', 'W41'), ('M12', 'W35'), ('M13', 'W44'), ('M14', 'W44'), ('M17', 'W38'), ('M17', 'W39'), ('M17', 'W6'), ('M19', 'W35'), ('M21', 'W8'), ('M23', 'W41'), ('M24', 'W44'), ('M24', 'W22'), ('M25', 'W16'), ('M27', 'W15'), ('M28', 'W27'), ('M30', 'W6'), ('M31', 'W50'), ('M34', 'W8'), ('M34', 'W28'), ('M34', 'W6'), ('M34', 'W21'), ('M34', 'W2'), ('M35', 'W33'), ('M38', 'W34'), ('M39', 'W44'), ('M39', 'W25'), ('M40', 'W15'), ('M40', 'W35'), ('M40', 'W4'), ('M40', 'W44'), ('M41', 'W17'), ('M41', 'W49'), ('M41', 'W9'), ('M41', 'W38'), ('M43', 'W40'), ('M45', 'W49'), ('M45', 'W46'), ('M45', 'W44'), ('M45', 'W47'), ('M45', 'W27'), ('M45', 'W15'), ('M46', 'W28'), ('M46', 'W39'), ('M46', 'W14'), ('M46', 'W45'), ('M46', 'W2'), ('M46', 'W34'), ('M46', 'W27'), ('M46', 'W47'), ('M46', 'W49'), ('M4

In [13]:
def make_row(name, summary):
    total = summary.get("total_instances", 0)
    stable = summary.get("stable_count", 0)

    return {
        "Dataset": name,
        "Instances": total,
        "Parsed": summary.get("parsed_count", 0),
        "Valid": summary.get("valid_count", 0),
        "Stable": stable,
        "Exact Match": summary.get("exact_match_count", 0),
        "Stable (%)": round((stable / total) * 100, 1) if total > 0 else 0.0,
        "Avg Blocking Pairs": round(summary.get("avg_blocking_pairs", 0.0), 2),
    }

results_df = pd.DataFrame([
    make_row("5-IC (Easy)", gemini_summary_5_ic),
    make_row("20-IC (Medium)", gemini_summary_20_ic),
    make_row("50-IC (Hard)", gemini_summary_50_ic),
])

display(results_df.style.hide(axis="index"))

Dataset,Instances,Parsed,Valid,Stable,Exact Match,Stable (%),Avg Blocking Pairs
5-IC (Easy),5,5,5,5,5,100.000000,0.000000
20-IC (Medium),5,5,5,5,5,100.000000,0.000000
50-IC (Hard),1,1,1,0,0,0.000000,95.000000


We evaluated an advanced reasoning model (Gemini) on stable matching generation tasks across Easy (n = 10), Medium (n = 20), and Hard (n = 50) datasets constructed under the Impartial Culture (IC) setting. The model demonstrated strong performance on both Easy and Medium instances, successfully generating stable and exact matchings across all tested cases despite the increased randomness and complexity inherent in IC preferences. However, when scaling to the Hard dataset, although the model produced a valid one-to-one matching, the solution was highly unstable, containing a large number of blocking pairs (95). This indicates a breakdown in the model’s ability to enforce stability constraints in larger markets. This behavior aligns with prior findings that, while advanced LLMs can effectively solve small and medium matching markets, their performance deteriorates significantly for larger instances due to increased combinatorial complexity and the need for iterative algorithmic reasoning over longer preference lists.